In [ ]:
import torch
from torch import nn
from torch.optim import Adam

class Engine:
    def __init__(self, model, device):
        self.model = model.to(device)
        self.device = device

    def train_with_early_stopping(
        self,
        train_dataloader,
        valid_dataloader,
        optimizer,
        loss_fn,
        epochs,
        patience=5
    ):

        train_losses = []
        valid_losses = []
        best_loss = float('inf')
        no_improve_epochs = 0

        for epoch in range(epochs):
            print(f"Epoch {epoch + 1}/{epochs}")

            # Training phase
            self.model.train()
            train_loss = 0
            for batch in train_dataloader:
                inputs, labels = batch[0].to(self.device), batch[1].to(self.device)
                optimizer.zero_grad()
                outputs = self.model(inputs)
                loss = loss_fn(outputs, labels)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            train_loss /= len(train_dataloader)
            train_losses.append(train_loss)

            # Validation phase
            self.model.eval()
            valid_loss = 0
            with torch.no_grad():
                for batch in valid_dataloader:
                    inputs, labels = batch[0].to(self.device), batch[1].to(self.device)
                    outputs = self.model(inputs)
                    loss = loss_fn(outputs, labels)
                    valid_loss += loss.item()

            valid_loss /= len(valid_dataloader)
            valid_losses.append(valid_loss)

            print(f"Training Loss: {train_loss:.4f}, Validation Loss: {valid_loss:.4f}")

            # Early stopping check
            if valid_loss < best_loss:
                best_loss = valid_loss
                no_improve_epochs = 0
                torch.save(self.model.state_dict(), 'best_model.pth')
                print("Validation loss improved, saving model.")
            else:
                no_improve_epochs += 1
                print(f"No improvement for {no_improve_epochs} epoch(s).")

            if no_improve_epochs >= patience:
                print("Early stopping triggered.")
                break

            # Adjust learning rate
            for param_group in optimizer.param_groups:
                param_group['lr'] *= 0.99  # Reduce learning rate by 1%

        # Load the best model
        self.model.load_state_dict(torch.load('best_model.pth'))
        print("Loaded the best model.")

        return {
            'train_loss': train_losses,
            'valid_loss': valid_losses
        }


